# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations. For the iABA974 model, annotation of uniprot identifiers to the genes was made possible by using BLAST with different microorganisms, as *Xanthobacter* sp. SoF1 is not included in the KEGG and Uniprot database. As a consequence, the gene id mapping for both microorganisms are included in the initialization of the ActiveEnzymes information. 


In [1]:
import pandas as pd
import os
from cobra.io import load_json_model, read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1] == 'i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping, _parse_gpr
from Modules.utils.preprocessing import *

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_EnzymaticData_empty.xlsx')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_xau-xdi_250521.xlsx')
GOTENZYMES_FILE_PATH_XAU = os.path.join('Data','Databases', '250409_xau_gotenzymes.json')
GOTENZYMES_FILE_PATH_XDI = os.path.join('Data','Databases', '250519_xdi_gotenzymes.json')
MODEL_FILE_PATH = os.path.join('Models', '250521_iABA974_kegg_bjp.sbml')


Loading PAModelpy modules version 0.0.4.9
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-03


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iABA974_EnzymaticData_{formatted_date}.xlsx')

Relatively complicated regex because of different gene ids after BLAST

1. **AAFOLC_\d{5}**: Matches identifiers from the SoF1 genome
2. **|**: Alternation operator to separate different patterns.
3. **Xaut_\d{4}**: Matches the *Xanthobacter autotrophicus* gene ids
4. **|**: Alternation operator for the next part.
5. **EZH22_\d{5}**: Matches the *Xanthobacter diaforans* locus tags

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(?:AAFOLC_\d{5}|Xaut_\d{4}|EZH22_\d{5})'

def extract_locus_tags(text):
    if pd.isna(text):
        return text
    return re.findall(locustag_regex, text)

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [SoF1 GEM github](https://github.com/iAMB-RWTH-Aachen/SoF1-GEM)**

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `xau` (*Xanthobacter autotrophicus*) of `xdi` (*Xanthobacter diaforans *) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Go to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. 
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

## 1. Get information from the model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [4]:
print('mapping the ids from the iABA974 model')
model =read_sbml_model(MODEL_FILE_PATH)
id_mapper = create_id_mapper_from_model(model)
gene_id_mapper = create_genetokeggid_mapper(model).set_index('gene_id').to_dict()['kegg_gene_id']
id_mapper

mapping the ids from the iABA974 model
Mapped 693 of 1986 reactions to KEGG reaction IDs.
Mapped 506 out of 973 genes to a KEGG identifier


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,ec-code
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN
2,12DGR141tipp,[nan],[nan],,False,NaN,NaN
3,12DGR160tipp,[nan],[nan],,False,NaN,NaN
4,12DGR161tipp,[nan],[nan],,False,NaN,NaN
...,...,...,...,...,...,...,...
1981,LINO_6_DESA,"[C02050, C00005, C00080, [C00007, D00003]]","[C03035, C00006, [C00001, C01328, D03703, D062...",,False,R03814,1.14.19.3
1982,FACOAE1836Z9Z12Z,"[C03035, [C00001, C01328, D03703, D06249, D063...","[C00010, [C03035, D07213, C06426], C00080]",,False,R08181,3.1.2.2
1983,NIT,"[C00138, [C00697, D00083], [C00002, D08646], [...","[C00080, C00139, [C00008, G11113], [C00009, D0...",AAFOLC_05260 and AAFOLC_05265 and AAFOLC_05270...,False,R05185,1.18.6.1
1984,FERHYD,"[C00138, C00080]","[C00139, C00282]",AAFOLC_05725,True,R00019,1.12.7.2


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

Parse the BIGG gene-protein-reaction associations

In [5]:
#get a list of genes in a GPR
id_mapper_df = id_mapper.copy()
id_mapper_df['locus_tag'] = id_mapper_df['GPR'].apply(extract_locus_tags)

#make sure each gene and ec number has a separate row
id_mapper_df = id_mapper_df.explode('locus_tag', ignore_index=True)
id_mapper_df = id_mapper_df.explode('ec-code', ignore_index=True)
id_mapper_df = id_mapper_df.drop_duplicates(['rxn_id','ec-code','locus_tag']) #previous operations can result in duplicate entries

#map the locus tags to gene_ids if possible (also for the gpr relations)
for col in ['GPR', 'locus_tag']:
        id_mapper_df[col] = id_mapper_df[col].apply(
            lambda x: replace_locustags_in_text(str(x), gene_id_mapper) if pd.notnull(x) else x
        )
id_mapper_df

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,ec-code,locus_tag
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,NaN
2,12DGR141tipp,[nan],[nan],,False,NaN,NaN,NaN
3,12DGR160tipp,[nan],[nan],,False,NaN,NaN,NaN
4,12DGR161tipp,[nan],[nan],,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
4994,NIT,"[C00138, [C00697, D00083], [C00002, D08646], [...","[C00080, C00139, [C00008, G11113], [C00009, D0...",Xaut_0143 and Xaut_0089 and Xaut_0090 and Xaut...,False,R05185,1.18.6.1,Xaut_0096
4995,NIT,"[C00138, [C00697, D00083], [C00002, D08646], [...","[C00080, C00139, [C00008, G11113], [C00009, D0...",Xaut_0143 and Xaut_0089 and Xaut_0090 and Xaut...,False,R05185,1.18.6.1,AAFOLC_05305
4996,FERHYD,"[C00138, C00080]","[C00139, C00282]",Xaut_0343,True,R00019,1.12.7.2,Xaut_0343
4997,CYTHYD,"[[C00125, C00923, C01070], C00282]","[C00924, C00080]",Xaut_2173 and Xaut_2174,False,R04015,1.12.2.1,Xaut_2173


In [6]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['locus_tag'] = uniprot_df['Gene Names'].apply(extract_locus_tags)
uniprot_df = uniprot_df.explode('locus_tag', ignore_index=True).dropna(subset = ['locus_tag'])

uniprot_df = uniprot_df[['Entry','locus_tag', 'Mass', 'Length']].rename({'Entry': 'uniprot_id'}, axis=1)
uniprot_df

/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,uniprot_id,locus_tag,Mass,Length
0,A7IQH5,Xaut_5073,25473,255
1,Q56837,Xaut_4865,41690,376
2,Q56837,Xaut_5047,41690,376
3,Q56839,Xaut_4867,57348,523
4,Q56840,Xaut_4868,26143,250
...,...,...,...,...
37561,Q8RM05,Xaut_3508,68303,631
37576,Q93CI6,Xaut_3578,33279,311
37577,Q93CI7,Xaut_3577,55557,504
37578,Q93CI8,Xaut_3576,41442,385


### 2.3 Match Uniprot to BIGG data

In [7]:
# Merge first on locus_tag
id_mapper_with_protein = pd.merge(
    id_mapper_df, 
    uniprot_df, 
    on='locus_tag', 
    how='left'
).rename({'uniprot_id':'enzyme_id'}, axis=1)
id_mapper_with_protein

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,ec-code,locus_tag,enzyme_id,Mass,Length
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN
2,12DGR141tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN
3,12DGR160tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN
4,12DGR161tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
4551,NIT,"[C00138, [C00697, D00083], [C00002, D08646], [...","[C00080, C00139, [C00008, G11113], [C00009, D0...",Xaut_0143 and Xaut_0089 and Xaut_0090 and Xaut...,False,R05185,1.18.6.1,Xaut_0096,A7IBG2,10584.0,98.0
4552,NIT,"[C00138, [C00697, D00083], [C00002, D08646], [...","[C00080, C00139, [C00008, G11113], [C00009, D0...",Xaut_0143 and Xaut_0089 and Xaut_0090 and Xaut...,False,R05185,1.18.6.1,AAFOLC_05305,NaN,NaN,NaN
4553,FERHYD,"[C00138, C00080]","[C00139, C00282]",Xaut_0343,True,R00019,1.12.7.2,Xaut_0343,A7IC58,36042.0,339.0
4554,CYTHYD,"[[C00125, C00923, C01070], C00282]","[C00924, C00080]",Xaut_2173 and Xaut_2174,False,R04015,1.12.2.1,Xaut_2173,A7IHC4,40242.0,370.0


In [8]:
#sometimes the kegg id is a list, need to make sure each entry has its own row
id_mapper_final = id_mapper_with_protein.explode('kegg.reaction').rename({'ec-code':'EC'}, axis=1)

## 3. Match the BiGG model reaction and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction. As there are two different organisms used to create a namespace mapping to uniprot and KEGG, we also need the enzymatic parameter predictions from both organisms.

In [9]:
#get the json file from GotEnzymes for both organisms and parse into workable format
eco_enzymes_json_xau = pd.DataFrame(
    pd.read_json(GOTENZYMES_FILE_PATH_XAU).enzymes.to_list()
)
eco_enzymes_json_xdi = pd.DataFrame(
    pd.read_json(GOTENZYMES_FILE_PATH_XDI).enzymes.to_list()
)
eco_enzymes_df = pd.concat([eco_enzymes_json_xau, eco_enzymes_json_xdi], ignore_index=True)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,Xaut_0011,R10715,1.1.1.6,C00424,5.9418
1,Xaut_0011,R10715,1.1.1.6,C00546,8.5735
2,Xaut_0011,R10717,1.1.1.6,C05235,2.9614
3,Xaut_0011,R01034,1.1.1.6,C00184,4.4236
4,Xaut_0011,R01034,1.1.1.6,C00116,2.9077


In [10]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = map_kcat_values_to_reaction_protein_association(id_mapper=id_mapper_final,
                                                                     gotenzymes_df = eco_enzymes_df
                                                                    )

eco_enzymes_merged.dropna(subset = ['kcat_values'])

Index(['rxn_id', 'Reactants', 'Products', 'GPR', 'reversible', 'kegg.reaction',
       'EC', 'locus_tag', 'enzyme_id', 'Mass', 'Length'],
      dtype='object')
Mapped 477 out of 4958 kcat values to reactions based on kegg gene id
Mapped 15118 out of 4958 kcat values to reactions based on ec number
Index(['rxn_id', 'Reactants', 'Products', 'GPR', 'reversible', 'kegg.reaction',
       'EC', 'locus_tag', 'enzyme_id', 'Mass', 'Length', 'gene', 'reaction_id',
       'ec_number', 'compound', 'kcat_values'],
      dtype='object')
Final merged dataset has 11098 unique reactions with kcat values.


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,Mass,Length,gene,ec_number,compound,kcat_values
0,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0,220.0,EZH22_09440,2.8.3.6,C02232,63.0144
1,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0,220.0,EZH22_09440,2.8.3.6,C00846,42.6761
2,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0,220.0,EZH22_09440,2.8.3.6,C00042,0.1837
3,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0,220.0,EZH22_09440,2.8.3.6,C00091,33.0478
4,4HOXPACMON,"[[C00642, C13636], C00080, C00004, [C00007, D0...","[C01161, [C00001, C01328, D03703, D06249, D063...",Xaut_0377,False,R02698,1.14.14.9,A7IC92,54574.0,494.0,Xaut_0377,1.14.14.9,C00642,19.5363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14333,XAND,"[[C00001, C01328, D03703, D06249, D06322, D000...","[C00080, C00004, C00366]",Xaut_2917 or AAFOLC_10410 or EZH22_23995 or EZ...,False,R02103,1.17.1.4,A0A974SG06,17214.0,160.0,EZH22_16705,1.17.1.4,C00366,2.8635
14380,SBP,"[C00447, [C00001, C01328, D03703, D06249, D063...","[C05382, [C00009, D05467]]",Xaut_3520,True,R01845,3.1.3.37,A7IL56,34903.0,329.0,Xaut_3520,3.1.3.37,C00447,20.8071
14381,SBP,"[C00447, [C00001, C01328, D03703, D06249, D063...","[C05382, [C00009, D05467]]",Xaut_3520,True,R01845,3.1.3.37,A7IL56,34903.0,329.0,Xaut_3520,3.1.3.37,C05382,21.6291
14382,PRUK,"[[C00002, D08646], C00199]","[[C00008, G11113], C00080, C01182]",Xaut_1914,True,R01523,2.7.1.19,A7IGL6,32977.0,291.0,Xaut_1914,2.7.1.19,C00199,105.1046


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [12]:
# Set default values
eco_enzymes_parsed = assign_directionalities_for_kcat_relations(eco_enzymes_merged.copy())
eco_enzymes_parsed = assign_defaults_for_proteins_without_mapping(eco_enzymes_parsed)
eco_enzymes_parsed

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,molMass,Length,gene,kcat_values,direction
0,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0000,220.0,EZH22_09440,63.0144,b
1,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0000,220.0,EZH22_09440,42.6761,f
2,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0000,220.0,EZH22_09440,0.1837,b
3,3OADPCOAT,"[C00846, C00091]","[C02232, C00042]",Xaut_1120 and EZH22_09440,False,R02990,2.8.3.6,A0A974SK62,23430.0000,220.0,EZH22_09440,33.0478,f
4,4HOXPACMON,"[[C00642, C13636], C00080, C00004, [C00007, D0...","[C01161, [C00001, C01328, D03703, D06249, D063...",Xaut_0377,False,R02698,1.14.14.9,A7IC92,54574.0000,494.0,Xaut_0377,19.5363,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15197,YUMPS,"[[C03736, C00117], [C00106, D09776, D00027]]","[[C00001, C01328, D03703, D06249, D06322, D000...",AAFOLC_04045 or (AAFOLC_00720 and AAFOLC_06605),True,NaN,3.2.-.-;4.2.1.70,Enzyme_AAFOLC_00720,39959.4825,325.0,AAFOLC_00720,13.7000,b
15198,YUMPS,"[[C03736, C00117], [C00106, D09776, D00027]]","[[C00001, C01328, D03703, D06249, D06322, D000...",AAFOLC_04045 or (AAFOLC_00720 and AAFOLC_06605),True,NaN,4.2.1.70,Enzyme_AAFOLC_00720,39959.4825,325.0,AAFOLC_00720,13.7000,b
15199,YUMPS,"[[C03736, C00117], [C00106, D09776, D00027]]","[[C00001, C01328, D03703, D06249, D06322, D000...",AAFOLC_04045 or (AAFOLC_00720 and AAFOLC_06605),True,NaN,3.2.-.-|4.2.1.70,Enzyme_AAFOLC_06605,39959.4825,325.0,AAFOLC_06605,13.7000,b
15200,YUMPS,"[[C03736, C00117], [C00106, D09776, D00027]]","[[C00001, C01328, D03703, D06249, D06322, D000...",AAFOLC_04045 or (AAFOLC_00720 and AAFOLC_06605),True,NaN,3.2.-.-;4.2.1.70,Enzyme_AAFOLC_06605,39959.4825,325.0,AAFOLC_06605,13.7000,b


In [13]:
eco_enzymes_mapped = eco_enzymes_parsed.copy()
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_mapped, model)
# eco_enzymes_mapped.gene = [[g] for g in eco_enzymes_mapped.gene]

gene2protein = {replace_locustags_in_text(k, gene_id_mapper): v for k, v in gene2protein.items()}

# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_mapped, gene2protein).reset_index()

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

,index,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,molMass,Length,gene,kcat_values,direction
0,477,12DGR120tipp,[nan],[nan],Gene_12DGR120tipp,False,NaN,NaN,Enzyme_12DGR120tipp,39959.4825,325.0,[Gene_12DGR120tipp],13.7,f
1,478,12DGR140tipp,[nan],[nan],Gene_12DGR140tipp,False,NaN,NaN,Enzyme_12DGR140tipp,39959.4825,325.0,[Gene_12DGR140tipp],13.7,f
2,479,12DGR141tipp,[nan],[nan],Gene_12DGR141tipp,False,NaN,NaN,Enzyme_12DGR141tipp,39959.4825,325.0,[Gene_12DGR141tipp],13.7,f
3,480,12DGR160tipp,[nan],[nan],Gene_12DGR160tipp,False,NaN,NaN,Enzyme_12DGR160tipp,39959.4825,325.0,[Gene_12DGR160tipp],13.7,f
4,481,12DGR161tipp,[nan],[nan],Gene_12DGR161tipp,False,NaN,NaN,Enzyme_12DGR161tipp,39959.4825,325.0,[Gene_12DGR161tipp],13.7,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15859,14364,pentso3abcpp,"[[C00002, D08646], [C00001, C01328, D03703, D0...","[[C00008, G11113], C00080, nan, [C00009, D05467]]",(EZH22_13220 and AAFOLC_13370) or (EZH22_05895...,False,NaN,NaN,A0A974SLI1_Enzyme_AAFOLC_13370,65411.4825,561.0,"[EZH22_13220, AAFOLC_13370]",13.7,f
15860,14365,pentso3abcpp,"[[C00002, D08646], [C00001, C01328, D03703, D0...","[[C00008, G11113], C00080, nan, [C00009, D05467]]",(EZH22_13220 and AAFOLC_13370) or (EZH22_05895...,False,NaN,NaN,A0A974SLL7_Enzyme_AAFOLC_19670_Enzyme_AAFOLC_1...,108304.9650,909.0,"[EZH22_05895, AAFOLC_19670, AAFOLC_19675]",13.7,f
15861,14366,pentso3abcpp,"[[C00002, D08646], [C00001, C01328, D03703, D0...","[[C00008, G11113], C00080, nan, [C00009, D05467]]",(EZH22_13220 and AAFOLC_13370) or (EZH22_05895...,False,NaN,NaN,A0A974SLL7_Enzyme_AAFOLC_19670_Enzyme_AAFOLC_1...,108304.9650,909.0,"[EZH22_05895, AAFOLC_19670, AAFOLC_19675]",13.7,f
15862,14367,pentso3abcpp,"[[C00002, D08646], [C00001, C01328, D03703, D0...","[[C00008, G11113], C00080, nan, [C00009, D05467]]",(EZH22_13220 and AAFOLC_13370) or (EZH22_05895...,False,NaN,NaN,A0A974SLL7_Enzyme_AAFOLC_19670_Enzyme_AAFOLC_1...,108304.9650,909.0,"[EZH22_05895, AAFOLC_19670, AAFOLC_19675]",13.7,f


In [14]:
#save the dataframe
#drop duplicate entries. If there are duplicates, make sure the mean kcat value is used to parametrize
eco_enzymes_final = eco_enzymes_mapped.groupby(
    ['rxn_id', 'enzyme_id', 'direction'], 
    as_index=False
).agg({
    'kcat_values': 'mean', 
    **{col: 'first' for col in eco_enzymes_mapped.columns if col not in ['rxn_id', 'enzyme_id', 'direction', 'kcat']
      }
})
# eco_enzymes_final.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [15]:
# Get the information about the enzyme sectors
other_sectors = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = None)
del other_sectors['ActiveEnzymes']

In [16]:
# Save it to a new excel file
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_final.to_excel(writer, sheet_name='ActiveEnzymes', index = False)
    for sheet, df in other_sectors.items():
        if sheet == 'ExcessEnzymes': sheet = 'UnusedEnzyme'
        df.to_excel(writer, sheet_name=sheet, index = False)